# Read Partiels and SDIF tracks

Partial-tracking exports from Partiels (PM2), AudioSculpt and Loris land in a
`tracks` table. This notebook reads the cuts under `data/interop/`, overlays
the flute Partiels export on its spectrogram, compares the inharmonic cello
dedupe, then writes SDIF back out. See `data/interop/README.md` for provenance.
No prior tutorial is required.

In [ ]:
from pathlib import Path
import numpy as np
import audiospylt as asp

## Partiels JSON (flute, harmonic)

`read_partiels` reads a folder of `Group_1_Partial_*.json` files. The SuperVP f0
track is skipped. `partial_id` is PM2's harmonic number when the export carries
a description.

In [ ]:
flute = asp.read_partiels("../data/interop/partiels/flute_json")
print(f"{len(flute)} rows, {flute.track_id.nunique()} tracks, "
      f"{flute.time.min():.3f}–{flute.time.max():.3f} s")
print("partial_id:", sorted(flute.partial_id.dropna().unique().tolist()))
print(flute.attrs["meta"])
flute.head()

150 rows, 10 tracks, 1.097–1.400 s; harmonics 1, 2, 3 and 19. Meta lists the 4
files read, `skipped: ['Group_1_Feature_Scoring.json']`, and `channels: 2`.

## JSON vs CSV

The CSV export of the same cut keeps 6 significant digits and has no
description, so `partial_id` is missing. Frequencies differ by a fraction of a
cent.

In [ ]:
csv = asp.read_partiels("../data/interop/partiels/flute_csv")
print(f"csv: {len(csv)} rows, {csv.track_id.nunique()} tracks, "
      f"partial_id all missing: {bool(csv.partial_id.isna().all())}")
print(f"same track_id: {(flute.track_id.values == csv.track_id.values).all()}")
cents = 1200 * np.log2(flute.freq.values / csv.freq.values)
print(f"max |freq| difference: {np.nanmax(np.abs(cents)):.3f} cents")

## Octave jump in partial 1

Partial 1 (track 0) starts at 664–667 Hz for its first 3 frames, then drops to
322–369 Hz: PM2's harmonic follows the SuperVP f0, which reads an octave high
there. `max_step_cents=600` splits a track where the step exceeds that, so
partial 1 becomes 2 tracks (10 tracks become 13).

In [ ]:
p1 = flute[flute.partial_id == 1]
print("partial 1, first frames:")
print(p1.head(4)[["time", "freq", "track_id"]].to_string(index=False))
rest = p1.iloc[3:]
print(f"after that: {rest.freq.min():.0f}–{rest.freq.max():.0f} Hz")
split = asp.read_partiels("../data/interop/partiels/flute_json", max_step_cents=600)
print(f"max_step_cents=600: {split.track_id.nunique()} tracks "
      f"(partial 1 -> {split.loc[split.partial_id == 1, 'track_id'].nunique()} tracks)")

## Overlay on the flute spectrogram

The two channels of the recording differ by at most 2.4e-07, so
`to_mono` of the stereo load matches the channel-0 analysis. The tracks sit at
1.097–1.400 s; the trim below starts at 1.0 s, so their times are shifted by
that start before the overlay. Partial 19 has its median at 6307 Hz, far above
partials 1–3, so the frequency axis is logarithmic from 200 Hz to 13 kHz. The first frame
of each partial is about twice its median: all four start an octave high, as
partial 1 did above. A short marker view shows one point per partial present in
each PM2 analysis frame.

In [ ]:
x, sr = asp.load_audio("../samples/saariaho_flute.wav", mono=False)
print(f"channel difference max {np.max(np.abs(x[0] - x[1])):.1e}")
for pid, g in flute.sort_values("time").groupby("partial_id"):
    print(f"partial {pid}: first frame {g.freq.iloc[0]:.0f} Hz, median {g.freq.median():.0f} Hz")
xm = asp.to_mono(x)
excerpt = asp.trim(xm, sr, 1.0, 1.5)
shifted = flute.assign(time=flute["time"] - 1.0)
asp.nb.show(asp.viz.plot_spectrogram(asp.spectrogram(excerpt, sr), overlay=shifted,
                                     fmin=200, fmax=13000, freq_scale="log", audio=excerpt))
short = shifted[(shifted.time >= 0.10) & (shifted.time <= 0.25)]
print(f"marker span: {len(short)} points at {short.time.nunique()} frame times, "
      f"{short.track_id.nunique()} tracks")
asp.nb.show(asp.viz.plot_events(short, mode="markers"))

On the 0.10–0.25 s span the cell prints 72 points at 19 frame times (3–4
partials per frame): one marker per partial present in each analysis frame, not
a continuous line.

## Cello JSON (inharmonic template)

Inharmonic PM2 in Partiels copies points across slots. `dedupe=True` (the
default) drops those copies; `partial_id` here is the PM2 slot, not a harmonic
number.

In [ ]:
cello = asp.read_partiels("../data/interop/partiels/cello_json")
raw = asp.read_partiels("../data/interop/partiels/cello_json", dedupe=False)
print(f"dedupe=True:  {len(cello)} rows, {cello.track_id.nunique()} tracks, "
      f"duplicates_dropped={cello.attrs['meta']['duplicates_dropped']}")
print(f"dedupe=False: {len(raw)} rows, {raw.track_id.nunique()} tracks")
print(f"extra rows that repeat a (time, freq, amp) point: {len(raw) - len(cello)}")
print("partial_id (slots):", sorted(cello.partial_id.dropna().unique().tolist()))

136 rows in 41 tracks with 60 duplicates dropped; without dedupe, 196 rows in
48 tracks.

## AudioSculpt `1TRC`

`read_sdif` reads sinusoidal tracks. `attrs["meta"]` keeps the file's name-value
table; a stereo analysis is two streams.

In [ ]:
snare = asp.read_sdif("../data/interop/sdif/kraft_snare_mono-trc_45hz.sdif")
print(f"snare: {len(snare)} rows, {snare.track_id.nunique()} tracks, "
      f"{snare.time.min():.3f}–{snare.time.max():.3f} s, "
      f"{snare.freq.min():.1f}–{snare.freq.max():.1f} Hz")
print(f"table_name={snare.attrs['meta'].get('table_name')!r}, "
      f"written_by={snare.attrs['meta'].get('written_by')!r}")
drum0 = asp.read_sdif("../data/interop/sdif/pheek_bdrum-trc.sdif")
drum1 = asp.read_sdif("../data/interop/sdif/pheek_bdrum-trc.sdif", stream=1)
print(f"streams {drum0.attrs['meta']['streams']}: "
      f"stream 0 {drum0.track_id.nunique()} tracks / {len(drum0)} rows, "
      f"stream=1 {drum1.track_id.nunique()} / {len(drum1)}")

## Loris `RBEP`

Loris breakpoints carry a `bandwidth` column (noise share, 0–1). Many tracks
hold a single breakpoint.

In [ ]:
loris = asp.read_sdif("../data/interop/sdif/loris_flute_rbep.sdif")
print(f"{len(loris)} rows, {loris.track_id.nunique()} tracks, "
      f"{loris.time.min():.3f}–{loris.time.max():.3f} s")
print(f"bandwidth median {loris.bandwidth.median():.2f}; "
      f"single-breakpoint tracks {(loris.groupby('track_id').size() == 1).sum()}")
loris.head()

## Chord sequence

`read_chord_seq` turns an AudioSculpt chord-sequence analysis into an events
table: one row per partial, one onset per segment. Marker times sit in
`attrs["meta"]["markers"]`.

In [ ]:
chords = asp.read_chord_seq("../data/interop/sdif/kraft_snare_mono-cs.sdif")
markers = chords.attrs["meta"]["markers"]
print(f"{len(chords)} rows, {chords.time_start.nunique()} segments, "
      f"{len(markers)} markers, {markers[0]:.3f}–{markers[-1]:.3f} s")
print(asp.infer_kind(chords))
chords.head()

## Write SDIF

`write_sdif` writes `1TRC` frames. The Partiels flute cut is already on one
time grid, so it writes point for point and reads back with identical times and
frequencies; `partial_id` becomes `track_id + 1`. Loris breakpoints all have
different times, so without `hop=` each track gets a row at every frame time
inside its span (`docs/interop.md`); `hop=0.005` resamples onto a 5 ms grid.

In [ ]:
Path("sdif").mkdir(exist_ok=True)
flute_path = asp.write_sdif(flute, "sdif/flute_partiels.sdif")
back = asp.read_sdif(flute_path)
print(f"flute write {flute_path.stat().st_size} bytes; "
      f"times equal {np.array_equal(back.time.values, flute.time.values)}, "
      f"freq equal {np.array_equal(back.freq.values, flute.freq.values)}, "
      f"partial_id == track_id + 1 {(back.partial_id == back.track_id + 1).all()}")
src = Path("../data/interop/sdif/loris_flute_rbep.sdif")
print(f"loris source {src.stat().st_size} bytes")
p_fill = asp.write_sdif(loris, "sdif/loris_fill.sdif")
filled = asp.read_sdif(p_fill)
print(f"no hop: {len(filled)} rows, {p_fill.stat().st_size} bytes")
p_hop = asp.write_sdif(loris, "sdif/loris_hop.sdif", hop=0.005)
hopped = asp.read_sdif(p_hop)
print(f"hop=0.005: {len(hopped)} rows, {p_hop.stat().st_size} bytes")